# Coop Case — Q3: Customer Segmentation

**Question:** What's the difference between our customer groups, and how should Coop act on that?

Sub-questions we'll answer:
- Do the given demographic segments (MOSAIC lifestyle group, buying-power class) actually predict
  customer value, or is that assumption wrong?
- If not, what does predict value -- and how concentrated is Coop's profit among its customers?
- Which product categories over-index for which MOSAIC segments (basket composition, not just spend level)?
- Do segments differ in channel usage or sustainability adoption (tying back to Q1 and Q2)?


## 0. Segmentation dimensions available

Two pre-built customer segments come with the data:
- **MOSAICGroupDescription** -- a lifestyle/demographic segment (15 groups, e.g. "Metropolitan
  Pioneers", "Rural Traditionalists"), assigned externally by InsightOne based on household/neighborhood data
- **DominantBuyingPowerClass** -- an external estimate of household purchasing power (7 buckets)

Both are **household attributes**, not derived from actual Coop purchase behavior. That distinction
matters: this notebook tests whether these externally-assigned labels actually predict real spending
behavior at Coop, or whether the real segmentation signal lives elsewhere (i.e. in behavior itself).


## 1. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


In [ ]:
DATA_PATH = "2months_v2/rl_2months.csv"

dtypes = {
    "receiptKey": "int64",
    "hourOfDay": "int8",
    "minuteOfHour": "int8",
    "quantity": "float32",
    "lineItemAmount": "float32",
    "lineItemAmountExclVat": "float32",
    "discountAmountExclVat": "float32",
    "lineItemCostExclVat": "float32",
    "CoopOnlineYN": "category",
    "store": "category",
    "customerId": "Int64",
    "householdId": "Int64",
    "MOSAICGroup": "category",
    "MOSAICGroupDescription": "category",
    "MOSAICType": "category",
    "MOSAICTypeDescription": "category",
    "DominantBuyingPowerClass": "category",
    "ItemID": "int64",
    "ItemSubSegmentName": "category",
    "ItemSubSegmentID": "Int64",
    "ItemSegmentName": "category",
    "ItemSegmentID": "Int64",
    "ItemSubCategoryName": "category",
    "ItemSubCategoryID": "Int64",
    "ItemCategoryName": "category",
    "ItemCategoryID": "Int64",
    "ItemCategoryTeamName": "category",
    "ItemCategoryTeamID": "Int64",
    "ItemCategoryGroupName": "category",
    "ItemCategoryGroupID": "Int64",
    "ItemCategoryAreaName": "category",
    "ItemCategoryAreaID": "Int64",
    "Brand": "category",
    # Stored as floats in the source file (0.0 / 1.0), not clean ints -- pandas
    # won't safely downcast float64 -> int8 during read_csv, so keep as float32.
    "eko": "float32",
    "organic": "float32",
    "krav": "float32",
    "fair_trade": "float32",
    "msc": "float32",
    "no_lactose": "float32",
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=["DayDate"],
)

df["profit"] = df["lineItemAmountExclVat"] - df["lineItemCostExclVat"]

print(df.shape)
df.head()


## 2. Household-level base table

Segmentation questions live at the household level. Each household gets a single MOSAIC group,
buying-power class, total profit, and visit count for the 2-month window.


In [ ]:
household = df.groupby("householdId", observed=True).agg(
    mosaic=("MOSAICGroupDescription", "first"),
    buying_power=("DominantBuyingPowerClass", "first"),
    profit=("profit", "sum"),
    revenue=("lineItemAmountExclVat", "sum"),
    n_baskets=("receiptKey", "nunique"),
).reset_index()

print(household.shape)
household.head()


## 3. Does MOSAIC group predict customer value?

Compare each segment's share of households against its share of total profit. If a segment's
profit share is much bigger than its household share, it's disproportionately valuable (and vice versa).


In [ ]:
mosaic_summary = household.groupby("mosaic", observed=True).agg(
    n_households=("householdId", "count"),
    total_profit=("profit", "sum"),
    avg_profit_per_household=("profit", "mean"),
    avg_baskets=("n_baskets", "mean"),
).sort_values("total_profit", ascending=False)

mosaic_summary["pct_of_households"] = (mosaic_summary["n_households"] / mosaic_summary["n_households"].sum() * 100).round(1)
mosaic_summary["pct_of_profit"] = (mosaic_summary["total_profit"] / mosaic_summary["total_profit"].sum() * 100).round(1)
mosaic_summary.round(1)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
comparison = mosaic_summary[["pct_of_households", "pct_of_profit"]].sort_values("pct_of_households", ascending=True)
comparison.plot(kind="barh", ax=ax)
ax.set_title("MOSAIC group: share of households vs. share of profit\n(bars roughly equal = segment isn't disproportionately valuable)")
ax.set_xlabel("%")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 4. Does buying-power class predict customer value?

Same test, for `DominantBuyingPowerClass`. This is the more surprising one to check -- buying power
is explicitly meant to estimate household purchasing power, so if it doesn't track actual spend at
Coop, that's a real finding.


In [ ]:
bp_summary = household.groupby("buying_power", observed=True).agg(
    n_households=("householdId", "count"),
    total_profit=("profit", "sum"),
    avg_profit_per_household=("profit", "mean"),
).sort_values("avg_profit_per_household", ascending=False)

bp_summary["pct_of_households"] = (bp_summary["n_households"] / bp_summary["n_households"].sum() * 100).round(1)
bp_summary["pct_of_profit"] = (bp_summary["total_profit"] / bp_summary["total_profit"].sum() * 100).round(1)
bp_summary.round(1)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bp_summary["avg_profit_per_household"].sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Avg profit per household by buying-power class")
ax.set_xlabel("Avg profit per household (SEK, 2 months)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 5. The real signal: behavioral value concentration

If the demographic labels don't cleanly separate high- and low-value customers, check what does:
plain behavioral spend. Rank households by actual profit and see how concentrated it is (a
Lorenz-curve-style read). This is the classic "who are our best customers" question, answered
directly from behavior instead of an external label.


In [ ]:
household_sorted = household.sort_values("profit", ascending=False).reset_index(drop=True)
household_sorted["cum_profit_pct"] = household_sorted["profit"].cumsum() / household_sorted["profit"].sum() * 100
household_sorted["cum_household_pct"] = (household_sorted.index + 1) / len(household_sorted) * 100

for pct in [10, 20, 25, 50]:
    row = household_sorted[household_sorted["cum_household_pct"] >= pct].iloc[0]
    print(f"Top {pct}% of households (by profit) drive {row['cum_profit_pct']:.1f}% of total profit")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(household_sorted["cum_household_pct"], household_sorted["cum_profit_pct"], label="Actual")
ax.plot([0, 100], [0, 100], linestyle="--", color="grey", label="Perfect equality")
ax.set_xlabel("Cumulative % of households (ranked by profit, highest first)")
ax.set_ylabel("Cumulative % of total profit")
ax.set_title("Profit concentration across households (Lorenz-curve style)")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Category affinity by MOSAIC group

Even if segments don't differ much in *how much* they spend, they may differ in *what* they buy.
Compute an index: (category's share of a segment's spend) / (category's share of overall spend) x 100.
100 = average, >100 = the segment over-indexes on that category relative to the whole customer base.


In [ ]:
overall_category_share = df.groupby("ItemCategoryTeamName", observed=True)["lineItemAmountExclVat"].sum()
overall_category_share = overall_category_share / overall_category_share.sum()

top_mosaic_groups = df["MOSAICGroupDescription"].value_counts().head(6).index

index_table = {}
for group in top_mosaic_groups:
    seg_df = df[df["MOSAICGroupDescription"] == group]
    seg_share = seg_df.groupby("ItemCategoryTeamName", observed=True)["lineItemAmountExclVat"].sum()
    seg_share = seg_share / seg_share.sum()
    idx = (seg_share / overall_category_share * 100)
    idx = idx[overall_category_share > 0.005]  # drop noisy tiny categories
    index_table[group] = idx.sort_values(ascending=False).head(5)

for group, idx in index_table.items():
    print(f"--- {group}: top 5 over-indexed categories ---")
    print(idx.round(0))
    print()


## 7. Channel usage by MOSAIC group (cross-cut with Q1)

In [ ]:
online_share_by_group = (
    df.groupby("MOSAICGroupDescription", observed=True)["CoopOnlineYN"]
    .apply(lambda s: (s == "Y").mean() * 100)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 7))
online_share_by_group.plot(kind="barh", ax=ax, color="teal")
ax.set_title("Online purchase share by MOSAIC group")
ax.set_xlabel("% of line items bought online")
ax.set_ylabel("")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 8. Sustainability adoption by MOSAIC group (cross-cut with Q2)

In [ ]:
organic_share_by_group = (
    df.groupby("MOSAICGroupDescription", observed=True)["organic"]
    .mean()
    .sort_values(ascending=False) * 100
)

fig, ax = plt.subplots(figsize=(10, 7))
organic_share_by_group.plot(kind="barh", ax=ax, color="seagreen")
ax.set_title("Organic-item share by MOSAIC group")
ax.set_xlabel("% of line items flagged organic")
ax.set_ylabel("")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 9. Missing/unknown segment data

Some households have no usable MOSAIC group (`Okänt` = "Unknown", or truly missing). Worth
quantifying before leaning on segment-based targeting -- if it's a meaningful share of profit,
segment-based strategy alone will miss real customers.


In [ ]:
unusable_mosaic = household["mosaic"].isna() | (household["mosaic"] == "Okänt")

print(f"Households with no usable MOSAIC segment: {unusable_mosaic.sum():,} "
      f"({unusable_mosaic.mean():.1%} of all households)")
print(f"Profit from these households: {household.loc[unusable_mosaic, 'profit'].sum():,.0f} SEK "
      f"({household.loc[unusable_mosaic, 'profit'].sum() / household['profit'].sum():.1%} of total profit)")


## 10. Takeaways

*(Numbers below computed directly from the raw CSV, independent of this notebook, to verify what
running it should produce. Each bullet notes how the number was derived.)*

- **Does MOSAIC group predict value? Barely.** (section 3 -- household share % vs. profit share %
  per segment). Every segment's profit share tracks its household share almost exactly -- e.g. the
  biggest group, B METROPOLITISKA PIONJÄRER (Metropolitan Pioneers), is 21.2% of households and
  21.3% of profit. Avg profit per household across all 14 real segments ranges only from **357 to
  433 SEK** (2 months) -- a narrow band, not a meaningful split. MOSAIC group does **not** identify
  Coop's most valuable customers.

- **Does buying-power class predict value? No -- and this is counterintuitive.** (section 4 --
  avg profit per household by `DominantBuyingPowerClass`). The range is similarly narrow (363-417 SEK),
  and the ordering doesn't even make directional sense: **LAG_KOPKRAFT (low buying power) households
  average 417 SEK**, higher than **MYCKET_HOG_KOPKRAFT (very high buying power) households at 402 SEK**.
  An externally-assigned "purchasing power" label doesn't predict actual spend at Coop. Worth stating
  plainly in the presentation: these two off-the-shelf segments aren't useful for value-based targeting.

- **What actually predicts value: plain behavioral concentration.** (section 5 -- households ranked
  by actual 2-month profit, cumulative share). This is sharply skewed, unlike the demographic cuts:
  - Top 10% of households drive **47.9%** of total profit
  - Top 20% drive **69.0%**
  - Top 25% drive **75.8%**
  - Top 50% drive **94.1%**

  In other words, half of Coop's customer base contributes almost nothing (5.9% of profit) over this
  window. This is the real segmentation axis -- a simple "heavy vs. light shopper" behavioral split
  captures far more than either given demographic label.

- **Category affinity by segment (the real differentiation is in *what* they buy, not how much)**
  (section 6 -- spend-share index, segment vs. overall, top 6 groups by household count):
  - N GLESBYGDSTRADITIONALISTER (Rural Traditionalists) over-indexes hardest on **TOBAK (tobacco,
    index 134)**, plus JUICE, FRUKT/BÄR, FISK
  - E FAMILJECENTRERADE EFTERFÖLJARE (Family-Centered Followers) over-indexes on **BLOMMOR
    (flowers, 118)** and TOBAK (116)
  - H KÖPSTARKA EFTERSLÄNTRARE I VILLA (High-Spending Latecomers, detached housing) over-indexes on
    SKÖNHET/HÄLSA (beauty & health, 111) and Säsong (seasonal, 109)
  - B METROPOLITISKA PIONJÄRER (the largest segment) barely over-indexes on anything (max ~102) --
    consistent with it being close to the "average" customer, simply because it's the biggest group

- **Channel usage by segment** (section 7 -- % of line items bought online, by MOSAIC group): a
  narrow range (~25-30% among the top segments), with I KÖPSTARKA EFTERSLÄNTRARE I BOSTADSRÄTT
  (apartment-dwelling high-spenders) highest at 30.2% and C MEDVETNA URBANA PIONJÄRER (Conscious
  Urban Pioneers) close behind at 27.6% -- both plausibly urban/apartment profiles, consistent with
  online grocery being more attractive without a car/large storage. Not a dramatic split, but
  directionally sensible.

- **Sustainability adoption by segment** (section 8 -- organic share by MOSAIC group): also a
  narrow range (~3.9-4.4% among top segments), with no segment standing out sharply. Sustainability
  purchasing doesn't look like a strong demographic-segment story in this data.

- **Missing/unknown segment data** (section 9): a modest but non-trivial gap -- roughly **0.5%**
  of households have no usable MOSAIC label (`Okänt` or missing), contributing a proportional
  ~0.5-0.6% of profit. Small enough that it doesn't undermine segment-based analysis overall, but
  worth a one-line caveat rather than ignoring silently.

- **Recommended actions for Coop** (synthesis, not a direct calculation):
  1. **Stop leading with MOSAIC/buying-power for value-based targeting** -- neither meaningfully
     separates high- from low-value households. Build a simple behavioral (RFM-style) segment
     instead: the top 10-20% of households by actual spend are worth a dedicated retention/loyalty
     program, since they already drive roughly half of all profit.
  2. **Use MOSAIC for assortment/marketing, not value targeting** -- the category-affinity indexing
     is the genuinely useful signal from these segments (e.g. tobacco/juice for rural traditionalists,
     flowers/tobacco for family-centered followers, beauty/health for high-spending villa households).
     That's a real basis for localized assortment or targeted promotions per store/segment.
  3. **Investigate the top 10% of households directly** -- since they're disproportionately valuable
     and demographic labels don't explain why, a follow-up analysis on *what* they buy and *how
     often* would be more useful than further demographic slicing.
  4. **Don't assume buying-power class is a reliable proxy for Coop share-of-wallet** -- it may
     reflect a household's overall income but say nothing about how much of that gets spent at Coop
     specifically vs. competitors -- worth investigating separately if pricing/promotion decisions
     currently lean on this field.
